# Pipeline to convert GMW OpenStreetMap country boundaries into vector tiles

Source: `Lab/data/raw/gmw_openstreetmap_country_boundaries_20260624.gpkg`

**Outputs:**
- `countries_12miles_location_v4.json` / `.mbtiles` — all countries layer for Mapbox
- `new_territories_bo.csv` — 6 new territories for backoffice import (BLM, CCK, KWT, MNP, IOT, NRU)

In [ ]:
import os
import csv
import json
import uuid
from pathlib import Path

import geopandas
import pandas as pd
import pyproj
import subprocess
import logging
from typing import Union

from shapely.geometry import mapping, box
from shapely.ops import transform

### Setup

In [ ]:
WORK_DIR = Path(os.getcwd())
# Navigate to Lab/ from notebook directory (layers -> data_processing -> Lab)
LAB_DIR = WORK_DIR.parents[1] if "data_processing" in str(WORK_DIR) else WORK_DIR
BASE_DIR = LAB_DIR / "data"
logging.basicConfig(level=logging.INFO)
logging.info(f"Base dir: {BASE_DIR}")

input_dir = BASE_DIR / "raw"
output_dir = BASE_DIR / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

## Path setup
in_file = input_dir / "gmw_openstreetmap_country_boundaries_20260624.gpkg"
locations_csv = input_dir / "locations_merged.csv"
layer_name = "gmw_openstreetmap_country_boundaries_20260624"

assert in_file.exists(), f"Input file not found: {in_file}"
assert locations_csv.exists(), f"Locations CSV not found: {locations_csv}"

### Helper functions

In [ ]:
def mbtilesGeneration(
    data_path: Path, output_path: Union[Path, None] = None, update: bool = False
) -> Path:
    """
    Generate mbtiles from a vector data file using mapshaper + tippecanoe.
    
    Parameters
    ----------
    data_path : Path - The path to the vector data file.
    output_path : Path - The path to the output mbtiles file.
    update : bool, optional - If True, the output will be overwritten.
    
    Returns
    -------
    Path - The path to the generated mbtiles file.
    """
    try:
        assert data_path.exists(), "Data path does not exist."

        if not output_path:
            output_path = data_path.with_suffix(".mbtiles")

        if update or not output_path.exists():

            if data_path.suffix != ".json":
                CMD = f'mapshaper {data_path} -clean allow-overlaps rewind -o format=geojson {data_path.with_suffix(".json")} force'
                subprocess.run(CMD, shell=True, check=True)
                data_path = data_path.with_suffix(".json")

            assert data_path.suffix == ".json", "Data path must be a json file."

            logging.info("Creating mbtiles file...")

            subprocess.run(
                f"tippecanoe -zg -f -P -o {output_path} --extend-zooms-if-still-dropping {data_path}",
                shell=True,
                check=True,
            )

        return output_path

    except Exception as e:
        logging.error(e)
        return 1

In [ ]:
def generate_location_idn(iso: str) -> str:
    """Generate a location_idn from an ISO code using UUID5 with NAMESPACE_OID.
    
    This is the same scheme used for all existing location_idn values in the CSV.
    The result is deterministic: the same ISO always produces the same UUID.
    """
    return str(uuid.uuid5(uuid.NAMESPACE_OID, iso))

### Load and inspect source data

In [ ]:
gdf = geopandas.read_file(in_file, driver="GPKG", layer=layer_name)
print(f"Features: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")
gdf.head(3)

### Dissolve features with the same ISO code

Some ISO codes have multiple features (e.g. ATF has 5 islands, BES has 3).
Merge them into single MultiPolygon features so each ISO appears once.

In [ ]:
# Dissolve by ISO code, joining names for merged features
gdf_dissolved = gdf.dissolve(
    by="iso_code_3char",
    aggfunc={"name_en": ", ".join},
).reset_index()

# Deduplicate repeated names (e.g. "Sudan, Sudan" → "Sudan")
gdf_dissolved["name_en"] = gdf_dissolved["name_en"].apply(
    lambda x: ", ".join(dict.fromkeys(x.split(", ")))
)

print(f"Before dissolve: {len(gdf)} features")
print(f"After dissolve: {len(gdf_dissolved)} features ({len(gdf) - len(gdf_dissolved)} merged)")

# Show merged names
merged = gdf_dissolved[gdf_dissolved["name_en"].str.contains(", ")]
if len(merged):
    print(f"\n=== Merged names ===")
    print(merged[["iso_code_3char", "name_en"]].to_string(index=False))

### Match with locations_merged.csv and generate ids for unmatched

In [ ]:
# Load location_idn and id mapping from locations_merged.csv (country rows only)
iso_to_location = {}
max_id = 4688  # 4688 is assigned to worldwide in the backoffice
with open(locations_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        rid = int(row["id"])
        if rid > max_id:
            max_id = rid
        if row["type"] == "country":
            iso_to_location[row["iso"]] = {
                "location_idn": row["location_idn"],
                "id": row["id"],
                "name": row["name"],
            }

logging.info(f"Found {len(iso_to_location)} countries in locations_merged.csv")
logging.info(f"New ids will start from: {max_id + 1}")

# ISOs that were merged during dissolve — use CSV name for these
merged_isos = set(gdf["iso_code_3char"].value_counts()[lambda x: x > 1].index)

# Names for unmatched countries with merged geometries
UNMATCHED_NAMES = {
    "UMI": "United States Minor Outlying Islands",
}

# Build result: match with CSV where possible, generate new ids for unmatched
rows = []
next_id = max_id + 1
for _, row in gdf_dissolved.iterrows():
    iso = row["iso_code_3char"]
    match = iso_to_location.get(iso)
    if match:
        location_idn = match["location_idn"]
        loc_id = match["id"]
        # Use CSV name only for merged ISOs, otherwise keep GPKG name_en
        name = match["name"] if iso in merged_isos else row["name_en"]
    else:
        location_idn = generate_location_idn(iso)
        loc_id = next_id
        next_id += 1
        name = UNMATCHED_NAMES.get(iso, row["name_en"])
    rows.append({
        "iso": iso,
        "name": name,
        "type": "country",
        "location_idn": location_idn,
        "id": loc_id,
        "geometry": row["geometry"],
    })

result = geopandas.GeoDataFrame(rows, crs=gdf.crs)

matched = result["iso"].isin(iso_to_location.keys()).sum()
unmatched_df = result[~result["iso"].isin(iso_to_location.keys())]
logging.info(f"Matched: {matched}, Unmatched: {len(unmatched_df)}")

print(f"\n=== Unmatched countries ({len(unmatched_df)}) — new ids generated ===")
print(unmatched_df[["iso", "name", "id", "location_idn"]].sort_values("iso").to_string(index=False))

### Export to GeoJSON

In [ ]:
json_path = output_dir / "countries_12miles_location_v4.json"
result.to_file(json_path, driver="GeoJSON")
logging.info(f"GeoJSON saved to {json_path}")

### Generate mbtiles

In [ ]:
output_mbtiles = output_dir / "countries_12miles_location_v4.mbtiles"
mbtiles_result = mbtilesGeneration(json_path, output_path=output_mbtiles, update=True)
logging.info(f"mbtiles generated at: {mbtiles_result}")

### Generate backoffice CSV for new territories

Export only the 6 new countries using the IDs assigned by the backoffice database.

In [ ]:
def geom_to_json(geom):
    """Convert shapely geometry to JSON string matching backoffice format."""
    geojson = mapping(geom)
    return json.dumps(geojson, separators=(", ", ":"))

def compute_metrics(geom):
    """Project geometry to UTM and compute area_m2, perimeter_m, coast_length_m."""
    centroid = geom.centroid
    zone = int((centroid.x + 180) / 6) + 1
    hemisphere = "south" if centroid.y < 0 else "north"
    utm_crs = pyproj.CRS(f"+proj=utm +zone={zone} +{hemisphere} +datum=WGS84")
    project = pyproj.Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True).transform
    geom_projected = transform(project, geom)
    return geom_projected.area, geom_projected.length, geom_projected.length

# Only these 6 territories, with their backoffice-assigned IDs
TARGET_ISOS_IDS = {
    "BLM": 4690,
    "CCK": 4722,
    "KWT": 4723,
    "MNP": 4724,
    "IOT": 4725,
    "NRU": 4726,
}
TARGET_ISOS = list(TARGET_ISOS_IDS.keys())

# Filter to target countries only
unmatched = result[result["iso"].isin(TARGET_ISOS)].copy()
# Reorder to match TARGET_ISOS
unmatched = unmatched.set_index("iso").loc[TARGET_ISOS].reset_index()
logging.info(f"Generating backoffice CSV for {len(unmatched)} new territories")

# Build CSV rows
bo_rows = []
for _, row in unmatched.iterrows():
    geom = row["geometry"]
    minx, miny, maxx, maxy = geom.bounds
    bbox_geom = box(minx, miny, maxx, maxy)
    area_m2, perimeter_m, coast_length_m = compute_metrics(geom)

    bo_rows.append({
        "id": TARGET_ISOS_IDS[row["iso"]],
        "name": row["name"],
        "location_type": "country",
        "iso": row["iso"],
        "bounds": geom_to_json(bbox_geom),
        "geometry": geom_to_json(geom),
        "area_m2": area_m2,
        "perimeter_m": perimeter_m,
        "coast_length_m": coast_length_m,
        "location_id": row["location_idn"],
    })

# Write CSV
bo_csv_path = output_dir / "new_territories_bo.csv"
bo_df = pd.DataFrame(bo_rows)
bo_df.to_csv(bo_csv_path, index=False)
logging.info(f"Backoffice CSV saved to {bo_csv_path}")
print(f"\n{len(bo_rows)} new territories written")
print(bo_df[["id", "name", "iso", "area_m2", "perimeter_m", "coast_length_m"]].to_string(index=False))